<a href="https://colab.research.google.com/github/gunasekhara67-del/PROJECTS/blob/main/AI_Data_Quality_%26_Root_Cause_Analysis_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pandas numpy matplotlib seaborn google-genai gradio

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from getpass import getpass
import google.genai as genai

In [ ]:
GEMINI_API_KEY= getpass("Enter your API here: ")
genai.api_key = GEMINI_API_KEY

In [8]:
df =pd.read_csv('/content/fifa_world_cup_2026_player_performance.csv')
print(df)

      player_id       player_name  age nationality   team  jersey_number  \
0        P00055        Rodri Fati   26     Spanish  Spain              3   
1        P00070   Ansu Le Normand   19     Spanish  Spain             18   
2        P00066        Gavi Ramos   18     Spanish  Spain             14   
3        P00073     Pedro Cubarsi   20     Spanish  Spain             21   
4        P00059  Alvaro Oyarzabal   23     Spanish  Spain              7   
...         ...               ...  ...         ...    ...            ...   
54595    P00996       Amir Younis   28       Iraqi   Iraq              8   
54596    P01006  Safaa Al-Khafaji   20       Iraqi   Iraq             18   
54597    P01008     Ahmed Hussein   26       Iraqi   Iraq             20   
54598    P00991     Ibrahim Mhawi   30       Iraqi   Iraq              3   
54599    P00989  Mohannad Hussein   27       Iraqi   Iraq              1   

         position  height_cm  weight_kg preferred_foot  ... possession_impact  \
0     

In [9]:
def analyze_data_quality(file_path):
    if file_path is None:
        return None, "Please upload a CSV file.", pd.DataFrame()

    df=pd.read_csv(file_path)
    issues=[]

    for column in df.columns:
        count=int(df[column].isnull().sum())
        if count>0:
            issues.append({"Issue":"Missing Values","Column":column,"Count":count,"Severity":"Medium"})

    duplicate_rows=int(df.duplicated().sum())
    if duplicate_rows>0:
        issues.append({"Issue":"Duplicate Rows","Column":"All Columns","Count":duplicate_rows,"Severity":"High"})

    if "Order_ID" in df.columns:
        duplicate_ids=int(df["Order_ID"].duplicated().sum())
        if duplicate_ids>0:
            issues.append({"Issue":"Duplicate Order_ID","Column":"Order_ID","Count":duplicate_ids,"Severity":"High"})

    if "Quantity" in df.columns:
        quantity=pd.to_numeric(df["Quantity"],errors="coerce")
        invalid=int((quantity<=0).sum())
        if invalid>0:
            issues.append({"Issue":"Invalid Quantity","Column":"Quantity","Count":invalid,"Severity":"High"})

    if "Unit_Price" in df.columns:
        price=pd.to_numeric(df["Unit_Price"],errors="coerce")
        invalid=int((price<=0).sum())
        if invalid>0:
            issues.append({"Issue":"Invalid Unit Price","Column":"Unit_Price","Count":invalid,"Severity":"High"})

    if "Order_Date" in df.columns:
        parsed=pd.to_datetime(df["Order_Date"],errors="coerce")
        invalid=int(parsed.isnull().sum())
        if invalid>0:
            issues.append({"Issue":"Invalid Date","Column":"Order_Date","Count":invalid,"Severity":"Medium"})

    issues_df=pd.DataFrame(issues,columns=["Issue","Column","Count","Severity"])
    summary=f"Rows: {df.shape[0]} | Columns: {df.shape[1]} | Missing cells: {int(df.isnull().sum().sum())} | Duplicate rows: {duplicate_rows} | Issues: {len(issues_df)}"
    return df,summary,issues_df

In [10]:
def create_quality_charts(df,issues_df):
    charts=[]
    missing=df.isnull().sum()
    missing=missing[missing>0]

    if len(missing)>0:
        plt.figure(figsize=(8,4))
        sns.barplot(x=missing.index,y=missing.values)
        plt.title("Missing Values by Column")
        plt.xticks(rotation=30)
        plt.tight_layout()
        p="/content/project2_missing_values.png"
        plt.savefig(p,dpi=150)
        plt.close()
        charts.append(p)

    if len(issues_df)>0:
        counts=issues_df["Severity"].value_counts()
        plt.figure(figsize=(8,4))
        sns.barplot(x=counts.index,y=counts.values)
        plt.title("Data Quality Issues by Severity")
        plt.tight_layout()
        p="/content/project2_severity.png"
        plt.savefig(p,dpi=150)
        plt.close()
        charts.append(p)

        plt.figure(figsize=(6,4))
        counts.plot.pie(autopct="%1.0f%%",ylabel="")
        plt.title("Issue Severity Distribution")
        plt.tight_layout()
        p="/content/project2_pie.png"
        plt.savefig(p,dpi=150)
        plt.close()
        charts.append(p)

    if "Unit_Price" in df.columns:
        prices=pd.to_numeric(df["Unit_Price"],errors="coerce").dropna()
        if len(prices)>0:
            plt.figure(figsize=(7,4))
            sns.boxplot(x=prices)
            plt.title("Unit Price Outlier View")
            plt.tight_layout()
            p="/content/project2_boxplot.png"
            plt.savefig(p,dpi=150)
            plt.close()
            charts.append(p)
    return charts

In [11]:
def generate_root_cause(issues_df):
    if len(issues_df)==0:
        return "No major data-quality issues were detected."

    prompt=f'''
You are a Data Quality Root-Cause Analysis Copilot.
Use only the supplied quality report.
For every issue provide:
1. Possible root cause
2. Business impact
3. Corrective action
4. Preventive recommendation
Keep the answer practical and easy to understand.
Do not invent data.

Quality report:
{issues_df.to_dict(orient="records")}
'''
    response=client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

In [12]:
def gradio_project2(file_path):
    if file_path is None:
        return "Upload a CSV file.", pd.DataFrame(), pd.DataFrame(), "No analysis yet.", []

    df,summary,issues_df=analyze_data_quality(file_path)
    charts=create_quality_charts(df,issues_df)
    ai_text=generate_root_cause(issues_df)

    return summary,df.head(20),issues_df,ai_text,charts

with gr.Blocks(title="AI Data Quality & Root-Cause Analysis Copilot") as app:
    gr.Markdown("# AI Data Quality & Root-Cause Analysis Copilot")
    gr.Markdown("Upload a CSV, detect data-quality problems, view charts, and receive Gemini-assisted root-cause analysis.")

    file_input=gr.File(label="Upload CSV",file_types=[".csv"],type="filepath")
    analyze_button=gr.Button("Analyze Data Quality")

    summary_output=gr.Textbox(label="Quality Summary")
    preview_output=gr.Dataframe(label="Data Preview")
    issues_output=gr.Dataframe(label="Detected Issues")
    ai_output=gr.Markdown(label="Gemini Root-Cause Analysis")
    chart_output=gr.Gallery(label="Charts",columns=2,height="auto")

    analyze_button.click(
        gradio_project2,
        inputs=file_input,
        outputs=[summary_output,preview_output,issues_output,ai_output,chart_output]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1852786c7e12792ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
